# 05 — Campaign copilot: gate, writers, validator, fast paths (Day 5)

End-to-end evaluation of `packages/copilot/` against the live pipeline:

| § | What it measures |
|---|---|
| 1 | The Decisions API gate live: 3 named questions in one call, probabilities, latency |
| 2 | Writer provider comparison on one identical brief (template / ollama / neurometric): latency, tokens, validator verdict, eyeball |
| 3 | The 4 use cases end-to-end + the 15-case seed eval (fast-path accuracy, pass@1/2, p50/p95) |
| 4 | Validator torture test: hand-corrupted drafts — each mutation must be caught |
| 5 | Fast-path timing: decline / redirect / clarify answer from the gate alone (no writer) |

Research only — reused code lives in `packages/copilot/`. Measured numbers go to the
`docs/concepts.md` §7 findings ledger; the figure goes through `save_fig()` →
`docs/figures/day5_copilot_latency.png`. Request traces land in gitignored `traces/`.

In [ ]:
import json
import os
import pathlib
import re
import sys
import time
import warnings

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # variable NAMES in .env.example; values are never printed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from packages.copilot import run_request
from packages.copilot.planner import extract_scenario, get_context, run_tools
from packages.copilot.providers import (
    DecisionsClient,
    NeurometricWriter,
    OllamaWriter,
    ProviderError,
    TemplateWriter,
)
from packages.copilot.router import build_questions, decide_fast_path, parse_route
from packages.copilot.schemas import DraftRequest, Intent, load_config
from packages.copilot.validator import validate
from packages.plotting import save_fig

CONFIG = load_config()
TIMINGS = {"gate_s": [], "fast_path_s": [], "write_s": {}, "validate_s": [],
           "stages": [], "req_stages": {}, "brief_s": []}

print("repo root:        ", ROOT)
print("configured writer:", CONFIG.writer.provider)
print("router thresholds:", CONFIG.router.model_dump())
print("validator config: ", CONFIG.validator.model_dump())
env_names = (
    "AZURE_ENDPOINT", "AZURE_API_KEY", "AZURE_DEPLOYMENT_NAME",
    "NEUROMETRIC_BASE_URL", "NEUROMETRIC_API",
)
print("env present:      ", {n: bool(os.getenv(n)) for n in env_names})

## §1 — The gate live: one call, three named questions

The router asks every gate question in a **single** Decisions API request (design rule from
`docs/decision_api.md`: independent questions ride one call so the fast paths stay fast):

1. `intent` — a *choice* over the four use cases plus `out_of_scope` (the "other" escape),
2. `policy_risk` — a *noul* predicate: targeting/exclusion by a sensitive attribute or proxy,
3. `missing_inputs` — a *noul* predicate: is a required parameter absent?

Thresholds (`configs/copilot.yaml`) turn the probabilities into a fast path:
risk ≥ 0.70 → decline · out-of-scope or confidence < 0.50 → redirect · missing ≥ 0.60 → clarify.

In [ ]:
GATE_QUESTION = (
    "I have $500 to spend on contacts at $0.10 each - "
    "how deep should I target and what should I expect to earn?"
)
client = DecisionsClient.from_env()
questions = build_questions()
state = {
    "user_question": GATE_QUESTION,
    "available_context": (
        "Campaign copilot for uplift-based email targeting on the Hillstrom "
        "experiment; tools: optimize_budget, compare_scenarios, "
        "get_segment_profile, design_ab_test, get_model_card."
    ),
}

t0 = time.perf_counter()
body = client.decide(state, questions)
gate_s = time.perf_counter() - t0
TIMINGS["gate_s"].append(gate_s)
print(f"one call carrying {len(questions)} named questions: {gate_s:.2f}s")
print("usage:", body.get("usage"))

for name, ans in body["answers"].items():
    shown = {k: v for k, v in ans.items() if k != "type"}
    print(f"\n[{name}] type={ans.get('type')} -> {json.dumps(shown)[:400]}")

route = parse_route(body["answers"])
fast = decide_fast_path(route, CONFIG)
print("\nroute:", route.intent.value,
      f"conf={route.intent_confidence:.2f}",
      f"policy_risk={route.policy_risk:.2f}",
      f"missing={route.missing_inputs:.2f}")
print("fast path ->", fast)

## §2 — Writer provider comparison: same brief, three providers

One identical `DraftRequest` (scenario + tool outputs computed **once** by the planner) goes
through every writer, and the same V1–V4 battery judges every draft. The comparison is
*latency × tokens × validator verdict*, not vibes. `neurometric/goal-setter` is the only model
exposed on the NEUROMETRIC gateway — if its briefs fail often, ollama stays
the default writer on evidence).

In [ ]:
scenario = extract_scenario(GATE_QUESTION, Intent.PLAN, CONFIG)
t0 = time.perf_counter()
context = get_context()  # boot cost, once per process: champion + split rebuild
print(f"context boot: {time.perf_counter() - t0:.2f}s  (holdout n={context.n_test})")

t0 = time.perf_counter()
tool_runs = run_tools(scenario, context, CONFIG)
plan_s = time.perf_counter() - t0
TIMINGS["stages"].append({"stage": "plan", "s": plan_s})
print(f"plan: {len(tool_runs)} tools in {plan_s:.2f}s -> {[r.name for r in tool_runs]}")

request = DraftRequest(
    intent=Intent.PLAN, question=GATE_QUESTION, scenario=scenario, tool_runs=tool_runs
)

providers = [
    ("template", lambda: TemplateWriter()),
    ("ollama", lambda: OllamaWriter(
        model=CONFIG.writer.ollama_model,
        base_url=CONFIG.writer.ollama_base_url,
        timeout=CONFIG.writer.timeout_s,
    )),
    ("neurometric", lambda: NeurometricWriter.from_env(
        model=CONFIG.writer.neurometric_model, timeout=CONFIG.writer.timeout_s
    )),
]
rows, drafts = [], {}
for name, factory in providers:
    try:
        draft = factory().write(request)
    except ProviderError as err:
        rows.append({"provider": name,
                     "status": f"unavailable ({err.__class__.__name__})"})
        continue
    t0 = time.perf_counter()
    report = validate(draft.text, tool_runs, scenario, CONFIG.validator)
    TIMINGS["validate_s"].append(time.perf_counter() - t0)
    failed = [c.rule for c in report.failures]
    drafts[name] = draft.text
    TIMINGS["write_s"][name] = round(draft.latency_s, 2)
    rows.append({
        "provider": name,
        "latency_s": round(draft.latency_s, 2),
        "in_tokens": draft.input_tokens,
        "out_tokens": draft.output_tokens,
        "validator": "pass" if report.passed else f"FAIL {failed}",
        "citations": len(report.citations),
        "words": len(draft.text.split()),
    })

comparison = pd.DataFrame(rows)
comparison

In [ ]:
for name in ("template", "ollama", "neurometric"):
    if name in drafts:
        print("=" * 28, name, "=" * 28)
        print(drafts[name])
        print()

## §3 — The four use cases end-to-end

`run_request` runs the whole pipeline per question: live gate → Day-4 tools → configured
writer (ollama) → V1–V4 with one rewrite, and appends a JSONL row per stage to `traces/`.
The seed eval then scores all 15 labelled cases (fast-path accuracy, intent accuracy on briefs,
tool-call correctness, validator pass@1/2, p50/p95 latency).

In [ ]:
USE_CASES = [
    ("plan", GATE_QUESTION),
    ("what_if", "What if contact cost rises to $0.60?"),
    ("explain", "Who are the persuadables and who should we avoid targeting?"),
    ("test_design", "How long should we test? We see 2,000 customers a day."),
]

packages = []
for expected, question in USE_CASES:
    pkg = run_request(question, config=CONFIG)  # live gate + configured writer + trace
    packages.append(pkg)
    for stage, s in pkg.stage_latencies.items():
        TIMINGS["req_stages"].setdefault(stage, []).append(round(s, 3))
    TIMINGS["brief_s"].append(round(pkg.latency_s, 3))
    checks = {c.rule: c.passed for c in pkg.validation.checks} if pkg.validation else {}
    intent_mark = "OK" if pkg.intent.value == expected else f"got {pkg.intent.value}"
    print(f"[{expected}] fast={pkg.fast_path} via {pkg.provider} "
          f"attempts={pkg.draft_attempts} degraded={pkg.degraded} "
          f"lat={pkg.latency_s:.1f}s intent={intent_mark}")
    print(f"   stages={ {k: round(v, 2) for k, v in pkg.stage_latencies.items()} } "
          f"checks={checks} citations={len(pkg.citations)} trace={pkg.trace_id}")

print("\n---- plan brief (verbatim) ----\n")
print(packages[0].prose)

In [ ]:
from evals.run_seed_eval import run_seed_eval

seed_report = run_seed_eval()  # 15 labelled cases, live gate, configured writer
seed_df = pd.DataFrame(seed_report["cases"])
SEED_SUMMARY = seed_report["summary"]
print(json.dumps(SEED_SUMMARY, indent=2))
seed_df[["id", "fast_path", "expected_fast_path", "fast_path_ok", "intent",
         "intent_ok", "tools_ok", "attempts", "degraded", "latency_s"]]

## §4 — Validator torture test

The validator is the safety lever (approach §3: tools compute, the writer narrates,
the **validator decides**), so it gets adversarially poked: take the passing template draft
and corrupt it five ways. Each mutation must flip exactly the rule that owns it —
V1 number-trace, V2 units, V3 causal language, V4 sections.

In [ ]:
good_text = drafts.get("template")
assert good_text, "template draft missing - rerun section 2"
base = validate(good_text, tool_runs, scenario, CONFIG.validator)
assert base.passed, [c.detail for c in base.failures]

block = re.search(r"## Assumptions.*?(?=\n## |\Z)", good_text, re.S)
assert block, "template should carry an Assumptions section"

MUTATIONS = {
    "invented dollar amount": good_text + "\nExpected profit: **$9,999** next quarter.",
    "foreign currency (rupee)": good_text + "\nBudget alternative: \u20b9500 instead.",
    "causal overclaim": good_text + "\nThese customers will buy anyway.",
    "deleted Assumptions section": good_text[:block.start()] + good_text[block.end():],
    "misquoted depth": good_text + "\nWe target the top 51% instead.",
}

rows = []
for label, corrupted in MUTATIONS.items():
    report = validate(corrupted, tool_runs, scenario, CONFIG.validator)
    failed = [c.rule for c in report.failures]
    caught = not report.passed
    rows.append({"mutation": label, "caught": caught,
                 "by": ", ".join(failed) or "NOT CAUGHT"})
torture = pd.DataFrame(rows)
assert torture["caught"].all(), torture[~torture["caught"]]
torture

## §5 — Fast-path timing

Decline / redirect / clarify are gate-only: **no tools, no writer, zero writer tokens**.
The only latency is the Decisions call itself. The original design claim was "<1s"; this cell
measures it honestly on today's endpoint (per the `docs/concepts.md` policy, the ledger records
what was measured, not what was promised).

In [ ]:
FAST_PATHS = [
    ("decline", "Only target women aged 25-34 for this offer."),
    ("redirect", "Write me a poem about spring."),
    ("clarify", "Should we run a campaign?"),
]
fast_rows = []
for expected, question in FAST_PATHS:
    t0 = time.perf_counter()
    pkg = run_request(question, config=CONFIG)  # gate only - returns before any tool/writer
    dt = time.perf_counter() - t0
    TIMINGS["fast_path_s"].append(round(dt, 3))
    TIMINGS["req_stages"].setdefault("gate", []).append(
        round(pkg.stage_latencies.get("gate", 0), 3))
    fast_rows.append({
        "expected": expected,
        "got": pkg.fast_path,
        "match": pkg.fast_path == expected,
        "gate_s": round(pkg.stage_latencies.get("gate", 0), 2),
        "total_s": round(dt, 2),
        "writer_tokens": 0,
    })
pd.DataFrame(fast_rows)

## Figure — per-stage latency

Deterministic stages (plan tools, validate, template) versus generative stages (the two LLM
writers) in one picture. Bars are medians of every measurement taken above.

In [ ]:
def _med(values):
    return float(np.median(values)) if values else float("nan")


req = TIMINGS["req_stages"]
series = {
    "gate\n(3 questions)": _med(req.get("gate", []) + TIMINGS["gate_s"]),
    "plan\n(Day-4 tools)": _med(req.get("plan", [])),
    "write:\ntemplate": TIMINGS["write_s"].get("template", float("nan")),
    "write:\nollama": TIMINGS["write_s"].get("ollama", float("nan")),
    "write:\nneurometric": TIMINGS["write_s"].get("neurometric", float("nan")),
    "validate\n(V1-V4)": _med(TIMINGS["validate_s"]),
    "brief total\n(full request)": _med(TIMINGS["brief_s"]),
    "fast path total\n(no writer)": _med(TIMINGS["fast_path_s"]),
}
colors = ["C0", "C0", "0.75", "C2", "C1", "0.75", "C3", "C4"]

fig, ax = plt.subplots(figsize=(10, 4.5))
names, values = list(series), list(series.values())
ax.bar(names, values, color=colors, edgecolor="black", lw=0.6)
for i, v in enumerate(values):
    ax.text(i, v, f"{v:.2f}s", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("seconds (median)")
ax.set_title("Campaign copilot per-stage latency - live, Day 5 (Hillstrom holdout)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "day5_copilot_latency")

## Findings → `docs/concepts.md` §7 ledger

The numbers measured above are written into the findings ledger **in the same Day-5 docs
commit**:

- gate latency (one call, three named questions) and gate-only fast-path totals,
- writer latency per provider + validator verdict per draft,
- seed-eval fast-path accuracy, validator pass@1 / pass@2, p50/p95 latency,
- provider verdict (which writer is the configured default, and why).

Worth noting for Day 6: the trace rows under `traces/` use the same JSONL schema the
Day-6 eval runner reads — no second telemetry system.